# ShadowLab Engine (Ollama Edition)
This notebook runs a headless Ollama server optimized for a T4 GPU. It automatically mounts your Google Drive, downloads the DeepHat model if missing, and exposes an Anthropic-compatible API via a Cloudflare Tunnel.

**Requirements:** Make sure your runtime is set to **T4 GPU**.

In [ ]:
# Cell 1: Mount Google Drive & Ensure Model Exists
import os
import subprocess
from google.colab import drive

print("🟢 Mounting Google Drive...")
drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/ShadowLab/models"
MODEL_PATH = f"{MODEL_DIR}/DeepHat-V1-7B.Q4_K_M.gguf"

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print(f"🟡 Model not found at {MODEL_PATH}. Downloading...")
    # Using wget as fallback if huggingface-cli isn't configured
    # Note: Replace with actual direct URL if this is a private model
    url = "https://huggingface.co/TheBloke/Llama-2-7b-Chat-GGUF/resolve/main/llama-2-7b-chat.Q4_K_M.gguf"
    subprocess.run(["wget", "-q", "--show-progress", "-O", MODEL_PATH, url])
    print("✅ Download complete.")
else:
    print(f"✅ Model found at {MODEL_PATH}")

In [ ]:
# Cell 2: Install Ollama
print("?? Installing dependencies...")
!apt-get update && apt-get install -y zstd pciutils lshw
print("?? Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# Cell 3: Start Ollama Server (Background)
import subprocess
import time
import os
import requests

# VRAM Optimizations for T4 GPU
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"

print("🟢 Starting Ollama server...")
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ
)

time.sleep(3)
try:
    r = requests.get("http://localhost:11434/api/version")
    print(f"✅ Ollama {r.json()['version']} is running in the background.")
except Exception as e:
    print("❌ Failed to start Ollama. Check /tmp/ollama.log")
    print(e)

In [ ]:
# Cell 4: Register Models
import subprocess

print("🟢 Creating DeepHat model from GGUF...")
modelfile_content = f"""FROM /content/drive/MyDrive/ShadowLab/models/DeepHat-V1-7B.Q4_K_M.gguf
PARAMETER num_gpu 99
PARAMETER num_ctx 8192
SYSTEM "You are DeepHat, a cybersecurity analysis assistant."
"""
with open("/tmp/Modelfile.deephat", "w") as f:
    f.write(modelfile_content)

subprocess.run(["ollama", "create", "deephat", "-f", "/tmp/Modelfile.deephat"])

print("🟢 Pulling Qwen2.5-VL...")
subprocess.run(["ollama", "pull", "qwen2.5vl:7b"])

print("\n✅ Available Models:")
!ollama list

In [ ]:
# Cell 5: Start Cloudflare Tunnel
import subprocess
import time
import re
import os

print("?? Downloading Cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("?? Starting Tunnel...")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=open("/tmp/cloudflared.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8) # Wait a bit longer for the tunnel to establish

print("\n=========================================")
print("? TUNNEL URL (COPY THIS):")
try:
    with open('/tmp/cloudflared.log', 'r') as f:
        log_content = f.read()
        urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
        if urls:
            tunnel_url = urls[0]
            print(tunnel_url)
            
            # Write to Google Drive for local Windows sync script
            sync_dir = "/content/drive/MyDrive/shadowlab"
            os.makedirs(sync_dir, exist_ok=True)
            with open(f"{sync_dir}/tunnel_url.txt", "w") as sync_file:
                sync_file.write(tunnel_url)
            print(f"\n?? URL saved to {sync_dir}/tunnel_url.txt for auto-sync!")
        else:
            print("Tunnel URL not found in logs yet. Printing raw logs for debugging:")
            print(log_content)
except Exception as e:
    print(f"Error reading logs: {e}")
print("=========================================")


In [ ]:
# Cell 6: Monitoring Keep-Alive
import time
import requests
import subprocess

print("🟢 Monitoring started. Keep this cell running.")
while True:
    try:
        r = requests.get("http://localhost:11434/api/version", timeout=10)
        gpu = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
            text=True
        ).strip()
        mem_used, mem_total, temp = gpu.split(", ")
        status = "🟢" if r.status_code == 200 else "🟡"
        print(f"[{time.strftime('%H:%M:%S')}] {status} API: {r.status_code} | GPU: {mem_used}/{mem_total} MB | Temp: {temp}°C")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] ❌ Health check failed: {e}")
    time.sleep(60)